# Lab4-Assignment about Named Entity Recognition and Classification

This notebook describes the assignment of Lab 4 of the text mining course. We assume you have succesfully completed Lab1, Lab2 and Lab3 as welll. Especially Lab2 is important for completing this assignment.

**Learning goals**
* going from linguistic input format to representing it in a feature space
* working with pretrained word embeddings
* train a supervised classifier (SVM)
* evaluate a supervised classifier (SVM)
* learn how to interpret the system output and the evaluation results
* be able to propose future improvements based on the observed results


## Credits
This notebook was originally created by [Marten Postma](https://martenpostma.github.io) and [Filip Ilievski](http://ilievski.nl) and adapted by Piek vossen

## [Points: 18] Exercise 1 (NERC): Training and evaluating an SVM using CoNLL-2003

**[4 point] a) Load the CoNLL-2003 training data using the *ConllCorpusReader* and create for both *train.txt* and *test.txt*:**

    [2 points]  -a list of dictionaries representing the features for each training instances, e..g,
    ```
    [
    {'words': 'EU', 'pos': 'NNP'}, 
    {'words': 'rejects', 'pos': 'VBZ'},
    ...
    ]
    ```

    [2 points] -the NERC labels associated with each training instance, e.g.,
    dictionaries, e.g.,
    ```
    [
    'B-ORG', 
    'O',
    ....
    ]
    ```

In [1]:
from nltk.corpus.reader import ConllCorpusReader
import numpy as np
import gensim

word_embedding_model = gensim.models.KeyedVectors.load_word2vec_format(r"../Lab_2/Google_News_files/GoogleNews-vectors-negative300.bin.gz", binary = True)

### Adapt the path to point to the CONLL2003 folder on your local machine
train = ConllCorpusReader('CONLL2003', 'train.txt', ['words', 'pos', 'ignore', 'chunk'])
training_features = []
training_gold_labels = []


for token, pos, ne_label in train.iob_words():
    a_dict = {
        "Word": token,
        "POS": pos,
        "Is_upper": token.isupper(),
        "Is_lower": token.islower(),
        "Is_title": token.istitle(),
        "Is_number": token.isdigit(),
       # add features
    }
    
    training_features.append(a_dict)
    training_gold_labels.append(ne_label)


valid_tokens = [(token, ne_label) for token, pos, ne_label in train.iob_words() if token != '' and token != 'DOCSTART']

num_tokens = len(valid_tokens)
training_vectors = np.zeros((num_tokens, 300))

for i, (token, ne_label) in enumerate(valid_tokens):
    if token in word_embedding_model:
        training_vectors[i] = word_embedding_model[token]
print(training_vectors[0])
print(training_gold_labels[0], training_gold_labels[1], training_gold_labels[2])

[ 3.73535156e-02 -2.03125000e-01  2.12890625e-01  2.44140625e-01
 -2.85156250e-01 -3.44238281e-02  6.68945312e-02 -1.87500000e-01
 -3.90625000e-02  8.48388672e-03 -2.89062500e-01 -8.34960938e-02
  9.08203125e-02 -2.73437500e-01 -3.92578125e-01 -1.06445312e-01
 -6.59179688e-02 -9.94873047e-03 -5.41992188e-02 -4.17480469e-02
  2.63671875e-01  7.95898438e-02  1.50390625e-01  1.94335938e-01
  2.12890625e-01  9.86328125e-02 -3.35937500e-01  1.58203125e-01
  2.83203125e-01  2.33398438e-01 -1.19140625e-01 -2.30468750e-01
  2.61718750e-01  5.95703125e-02  2.61230469e-02 -3.41796875e-01
 -1.54296875e-01  1.37695312e-01  9.86328125e-02  5.56640625e-02
  3.14453125e-01  9.81445312e-02  1.58203125e-01  1.97265625e-01
  2.27050781e-02 -7.61718750e-02 -2.96875000e-01  2.18750000e-01
 -3.59375000e-01  1.88476562e-01 -1.08398438e-01  3.15856934e-03
 -5.83496094e-02  1.96289062e-01  1.28906250e-01 -2.31445312e-01
 -3.92578125e-01  1.36108398e-02 -2.94921875e-01 -7.76367188e-02
 -1.85546875e-01 -2.98828

In [22]:
### Adapt the path to point to the CONLL2003 folder on your local machine
train = ConllCorpusReader('CONLL2003', 'test.txt', ['words', 'pos', 'ignore', 'chunk'])

test_features = []
test_gold_labels = []

for token, pos, ne_label in train.iob_words():
    a_dict = {
        "Word": token,
        "POS": pos,
        "Is_upper": token.isupper(),
        "Is_lower": token.islower(),
        "Is_title": token.istitle(),
        "Is_number": token.isdigit(),
        # add features
    }
    
    test_features.append(a_dict)
    test_gold_labels.append(ne_label)
    
valid_tokens = [(token, ne_label) for token, pos, ne_label in train.iob_words() if token != '' and token != 'DOCSTART']

num_tokens = len(valid_tokens)
testing_vectors = np.zeros((num_tokens, 300))

for i, (token, ne_label) in enumerate(valid_tokens):
    if token in word_embedding_model:
        testing_vectors[i] = word_embedding_model[token]
print(testing_vectors[0], training_vectors.shape, testing_vectors.shape)
print(test_gold_labels[0], test_gold_labels[1], test_gold_labels[2])

[ 1.26953125e-01  2.60009766e-02  2.69531250e-01 -1.32812500e-01
  5.05371094e-02  8.11767578e-03 -1.33789062e-01 -2.91015625e-01
 -2.53906250e-01  2.81250000e-01 -9.91210938e-02 -5.22460938e-02
 -4.88281250e-01  1.12792969e-01 -2.66113281e-02  2.81250000e-01
  2.67578125e-01  3.37890625e-01 -1.77734375e-01  5.76171875e-02
 -1.06445312e-01  3.06640625e-01  3.33984375e-01 -1.85546875e-01
 -1.47705078e-02  6.15234375e-02  2.65625000e-01  3.04687500e-01
  2.41210938e-01  3.02734375e-01  4.46777344e-02  5.73730469e-02
 -1.57226562e-01 -6.64062500e-01 -1.26953125e-01 -1.42578125e-01
 -5.71289062e-02  1.92260742e-03 -1.06933594e-01  1.81884766e-02
 -3.24218750e-01 -6.99218750e-01  1.32812500e-01  6.86645508e-05
  1.62109375e-01 -3.75000000e-01  2.59765625e-01  5.22460938e-02
  1.68457031e-02  3.98437500e-01 -2.08984375e-01  3.53515625e-01
 -4.54101562e-02  1.57226562e-01 -6.05468750e-02 -1.96289062e-01
 -1.05468750e-01 -1.05957031e-01 -2.06298828e-02  4.27246094e-03
 -4.23828125e-01 -2.57812

In [3]:
print(training_features[0])

{'Word': 'EU', 'POS': 'NNP', 'Is_upper': True, 'Is_lower': False, 'Is_title': False, 'Is_number': False}


**[2 points] b) provide descriptive statistics about the training and test data:**
* How many instances are in train and test?
* Provide a frequency distribution of the NERC labels, i.e., how many times does each NERC label occur?
* Discuss to what extent the training and test data is balanced (equal amount of instances for each NERC label) and to what extent the training and test data differ?

Tip: you can use the following `Counter` functionality to generate frequency list of a list:

In [4]:
from collections import Counter


print(f'Length of training list: {len(training_features)}\n')
print(f'Length of test list: {len(test_features)}\n')
print(f'Frequency distribution of NERC labels in training list: {Counter(training_gold_labels)}\n')
print(f'Frequency distribution of NERC labels in training list: {Counter(test_gold_labels)}')

Length of training list: 203621

Length of test list: 46435

Frequency distribution of NERC labels in training list: Counter({'O': 169578, 'B-LOC': 7140, 'B-PER': 6600, 'B-ORG': 6321, 'I-PER': 4528, 'I-ORG': 3704, 'B-MISC': 3438, 'I-LOC': 1157, 'I-MISC': 1155})

Frequency distribution of NERC labels in training list: Counter({'O': 38323, 'B-LOC': 1668, 'B-ORG': 1661, 'B-PER': 1617, 'I-PER': 1156, 'I-ORG': 835, 'B-MISC': 702, 'I-LOC': 257, 'I-MISC': 216})


In [5]:
my_list=[1,2,1,3,2,5]
Counter(my_list)


Counter({1: 2, 2: 2, 3: 1, 5: 1})

**[2 points] c) Concatenate the train and test features (the list of dictionaries) into one list. Load it using the *DictVectorizer*. Afterwards, split it back to training and test.**

Tip: You’ve concatenated train and test into one list and then you’ve applied the DictVectorizer.
The order of the rows is maintained. You can hence use an index (number of training instances) to split the_array back into train and test. Do NOT use: `
from sklearn.model_selection import train_test_split` here.


In [6]:
from sklearn.feature_extraction import DictVectorizer

In [17]:
vec = DictVectorizer(sparse=False)

# Concatenate training and test features
all_features = training_features + test_features

# Apply vectorization
all_feature_matrix = vec.fit_transform(all_features)

# Split the vectorized features back into training and testing sets based on the number of training instances
num_train = len(training_features)
train_feature_matrix = all_feature_matrix[:num_train]
test_feature_matrix = all_feature_matrix[num_train:]

print(f"Train matrix: {train_feature_matrix}")
print(f"Test matrix: {test_feature_matrix}")
print()
print(f"Shape of training feature matrix: {train_feature_matrix.shape}")
print(f"Shape of test feature matrix: {test_feature_matrix.shape}")

Train matrix: [[0. 0. 0. ... 0. 0. 0.]
 [1. 0. 0. ... 0. 0. 0.]
 [0. 0. 1. ... 0. 0. 0.]
 ...
 [0. 1. 0. ... 0. 0. 0.]
 [0. 0. 1. ... 0. 0. 0.]
 [0. 1. 0. ... 0. 0. 0.]]
Test matrix: [[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 1. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]

Shape of training feature matrix: (203621, 27365)
Shape of test feature matrix: (46435, 27365)


**[4 points] d) Train the SVM using the train features and labels and evaluate on the test data. Provide a classification report (sklearn.metrics.classification_report).**
The train (*lin_clf.fit*) might take a while. On my computer, it took 1min 53s, which is acceptable. Training models normally takes much longer. If it takes more than 5 minutes, you can use a subset for training. Describe the results:
* Which NERC labels does the classifier perform well on? Why do you think this is the case?
* Which NERC labels does the classifier perform poorly on? Why do you think this is the case?

In [27]:
# Check the shapes of the feature matrices and labels
print(f"Shape of training feature matrix: {training_vectors.shape}")
print(f"Shape of training labels: {len(training_gold_labels)}")
print(f"Shape of testing feature matrix: {testing_vectors.shape}")
print(f"Shape of testing labels: {len(test_gold_labels)}")
print(f"Shape of training feature matrix: {train_feature_matrix.shape}")
print(f"Shape of testing feature matrix: {test_feature_matrix.shape}")


Shape of training feature matrix: (203621, 300)
Shape of training labels: 203621
Shape of testing feature matrix: (46435, 300)
Shape of testing labels: 46435
Shape of training feature matrix: (203621, 27365)
Shape of testing feature matrix: (46435, 27365)


In [28]:
from sklearn import svm
from sklearn_crfsuite.metrics import flat_classification_report

In [29]:
lin_clf = svm.LinearSVC()

In [30]:
lin_clf.fit(train_feature_matrix, training_gold_labels)
test_predictions = lin_clf.predict(test_feature_matrix)

print(flat_classification_report(test_gold_labels, test_predictions))

/Users/robertostoica/Desktop/anaconda3/envs/Text_Mining_python/lib/python3.12/site-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


ValueError: Found input variables with inconsistent numbers of samples: [79801, 78336]

**[6 points] e) Train a model that uses the embeddings of these words as inputs. Test again on the same data as in 2d. Generate a classification report and compare the results with the classifier you built in 2d.**

In [10]:
# your code here

## [Points: 10] Exercise 2 (NERC): feature inspection using the [Annotated Corpus for Named Entity Recognition](https://www.kaggle.com/abhinavwalia95/entity-annotated-corpus)
**[6 points] a. Perform the same steps as in the previous exercise. Make sure you end up for both the training part (*df_train*) and the test part (*df_test*) with:**
* the features representation using **DictVectorizer**
* the NERC labels in a list

Please note that this is the same setup as in the previous exercise:
* load both train and test using:
    * list of dictionaries for features
    * list of NERC labels
* combine train and test features in a list and represent them using one hot encoding
* train using the training features and NERC labels

In [ ]:
import pandas

In [ ]:
##### Adapt the path to point to your local copy of NERC_datasets
path = 'nerc_datasets/ner_v2.csv'
kaggle_dataset = pandas.read_csv(path, error_bad_lines=False)

In [ ]:
len(kaggle_dataset)

In [ ]:
df_train = kaggle_dataset[:100000]
df_test = kaggle_dataset[100000:120000]
print(len(df_train), len(df_test))

**[4 points] b. Train and evaluate the model and provide the classification report:**
* use the SVM to predict NERC labels on the test data
* evaluate the performance of the SVM on the test data

Analyze the performance per NERC label.

## End of this notebook